# 13 物品图片预览与裁剪

## 1. 本节点目标
让用户在添加物品前像更换头像一样预览照片，通过拖动和缩放选择最终卡片画面；整个过程在本机完成，不使用付费图像服务。

## 2. 完成结果与验收
- 选择 JPG、PNG 或 WebP 后显示固定 1.16:1 预览框。
- 支持拖动、100% 到 300% 缩放、重置和确认构图。
- 未确认构图时添加按钮不可用；确认后显示最终预览。
- 服务端再次校验裁剪框，最终保存为 1160×1000 WebP。
- 图片不会发送给大模型或第三方图像服务。
- 全量 74 项自动化测试通过。

## 3. 本节点文件结构
- `src/smart_laundry/image_cropper_component.py`：浏览器中的预览画布、拖动和缩放。
- `src/smart_laundry/image_storage.py`：格式、大小、裁剪范围校验和 WebP 保存。
- `app.py`：上传、确认构图、最终预览和添加物品流程。
- `tests/test_image_storage.py`：裁剪尺寸、越界参数和保存结果测试。

## 4. 关键代码解释
浏览器只返回 `left`、`top`、`width`、`height` 四个 0 到 1 之间的比例值。Python 不直接相信这些值，而是再次检查是否越界，再换算为原图像素并裁剪。这样无论原图分辨率是多少，构图描述都保持一致。

In [ ]:
crop = {'left': 0.1, 'top': 0.2, 'width': 0.58, 'height': 0.5}
image_width, image_height = 1200, 900
pixel_box = (
    round(crop['left'] * image_width),
    round(crop['top'] * image_height),
    round((crop['left'] + crop['width']) * image_width),
    round((crop['top'] + crop['height']) * image_height),
)
pixel_box

## 5. 数据流
选择本地图片 → 校验格式和 5 MB 限制 → 浏览器画布预览 → 用户拖动或缩放 → 确认归一化裁剪框 → Python 校验范围 → 裁剪并压缩为固定尺寸 → SQLite 只记录本地文件路径。

## 6. 关键概念
- **cover**：让图片覆盖整个框，必要时裁掉边缘，避免出现空白。
- **归一化坐标**：用 0 到 1 的比例表示位置，与原图像素无关。
- **服务端校验**：即使浏览器已经限制过，Python 仍检查输入。
- **WebP**：在保持视觉质量时通常比原始大图更节省空间。

## 7. 为什么这样设计
项目复用物品卡片的 1.16:1 比例，让预览和最终结果一致。裁剪参数由前端计算、后端执行和校验，兼顾顺滑交互与数据安全。当前不保留原始大图，减少本地存储和隐私暴露；需要更换构图时重新选择照片即可。

## 8. 常见错误与排查
1. 看不到预览：确认文件格式受支持且不超过 5 MB。
2. 添加按钮不可用：先点击“确认使用这个构图”。
3. 拖动后出现空白：重置构图；组件会限制图片边缘不离开预览框。
4. 最终画面不对：点击“重新调整构图”再确认。
5. 手机拖不动：确认手指从图片预览框内部开始移动。

## 9. 面试可能追问
**问：为什么不直接保存前端裁剪后的图片？** 答：后端保存能统一验证大小、格式和裁剪范围，避免信任浏览器输入。

**问：为什么返回比例而不是像素？** 答：比例不依赖浏览器显示尺寸，可以准确映射回任何分辨率的原图。

**追问：AI 油画功能呢？** 答：当前版本明确不调用付费图像模型，只保留免费本地裁剪；未来可在用户授权和成本可控后接入独立 provider。

## 10. 必须掌握的最少知识
理解预览框和最终卡片比例相同；拖动改变画面位置，缩放改变裁剪范围；浏览器负责交互，Python 负责验证和保存；本功能不产生模型费用。

## 11. 可自测小题
1. 为什么使用归一化坐标？
2. 为什么后端还要验证裁剪框？
3. 未确认构图时为什么禁用添加按钮？
4. 图片会不会发送给大模型？

<details><summary>参考答案</summary>1. 不依赖原图和浏览器像素。2. 不能信任客户端输入。3. 防止保存与预览不一致的图片。4. 不会。</details>

## 12. 动手小练习
1. 分别上传横图和竖图，比较初始 cover 效果。
2. 放大到 200% 后移动主体，再确认最终预览。
3. 运行 `python -m pytest tests/test_image_storage.py`。

## 13. 本节点术语表
| 术语 | 简单解释 |
|---|---|
| crop | 裁掉预览框以外的部分 |
| canvas | 浏览器绘图画布 |
| aspect ratio | 宽度与高度的比例 |
| pointer event | 同时支持鼠标和触控的操作事件 |
| EXIF orientation | 手机照片记录的拍摄方向信息 |

## 14. 下一节点连接
当前图片流程已完成免费本地闭环。未来如果加入风格转换，应把它作为可选 provider，并继续保留原图确认、隐私说明、费用提示和失败时使用普通裁剪图的降级路径。